# PharmGuard — Colab Runner

Single Colab entry point for the PharmGuard pipeline. Each cell invokes one stage script. Re-running individual cells re-runs only that stage; everything else stays intact on Drive.

**Before running this notebook the first time:**

1. **Mount Drive.** Required for persistence across sessions.
2. **HF account setup (one-time, manual):**
    1. Sign in at https://huggingface.co
    2. Visit https://huggingface.co/datasets/jhlee0619/mpib and click **Agree and access repository**
    3. Create a Read token at https://huggingface.co/settings/tokens
    4. Run the `Cell B — HF token` cell below (you'll paste the token there)
3. **Verify you have GPU access:** `Runtime → Change runtime type → GPU` (L4 recommended).

## Cell A — Mount Drive and clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO_URL  = 'https://github.com/NehlTech/pharmGuard.git'
REPO_PATH = '/content/pharmguard'

if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(['git', '-C', REPO_PATH, 'pull'], check=True)

print(f'\n✓ Repo at {REPO_PATH}')

## Cell B — Save your HF token (run once per Drive)

In [ ]:
from getpass import getpass
from pathlib import Path

token_path = Path('/content/drive/MyDrive/pharmguard/.hf_token')
token_path.parent.mkdir(parents=True, exist_ok=True)

if token_path.exists():
    print(f'Token already saved at {token_path}')
    print('Re-run this cell only if you need to replace it.')
else:
    token = getpass('Paste your HF token (starts with hf_): ').strip()
    if not token.startswith('hf_'):
        raise ValueError("Token should start with 'hf_'")
    token_path.write_text(token)
    token_path.chmod(0o600)
    print(f'✓ Token saved to {token_path}')

## Stage 00 — Setup

In [ ]:
!cd /content/pharmguard && python scripts/00_setup.py

## Stage 01 — Pharmaceutical data acquisition

In [ ]:
!cd /content/pharmguard && python scripts/01_acquire_data.py

## Stage 02 — MPIB acquisition and EDA

In [ ]:
!cd /content/pharmguard && python scripts/02_acquire_mpib.py

## Stage 03 — Reconstruct redacted V2 payloads (R1–R10)

MPIB redacts every V2 payload (`[REDACTED_PAYLOAD]`). This stage generates rule-family-appropriate substitutes using Phi-3-mini-4k-instruct backed by hand-authored templates.

All ten rule families from MPIB Table 9 are implemented: R1 Evidence Exaggeration, R2 Contraindication Masking, R3 Subgroup Generalization, R4 Outdated-as-Latest, R5 Fabricated Citation, R6 Warning Demotion, R7 Editorial Note, R8 Triage Downplay, R9 Dose Tweak, R10 Provenance Spoofing.

Adds two flags worth knowing:
* `--no-llm` — skip Phi-3-mini, use only deterministic skeletons (smoke-test mode, ~30 seconds total)
* `--families R7,R8` — restrict to specific families

In [ ]:
!cd /content/pharmguard && python scripts/03_reconstruct_payloads.py

## Inspect the generated samples

Run this to see one reconstructed payload from each rule family (10 total). The orchestrator collects samples with family stratification, so every family is represented.

In [ ]:
import json
from pathlib import Path

samples_path = Path('/content/drive/MyDrive/pharmguard/logs/reconstruction_samples.json')
with samples_path.open() as f:
    samples = json.load(f)

for i, s in enumerate(samples, 1):
    print(f"{'=' * 70}")
    print(f"Sample {i}: {s['sample_id']}")
    print(f"  Rule family: {s['rule_family']}")
    print(f"  Status:      {s['status']}")
    print(f"  Target features: {s['target_features']}")
    print(f"  Actual features: {s['actual_features']}")
    print(f"  Deviation:       {s['feature_deviation']}")
    print()
    print(f"  Generated payload:")
    print(f"  {'-' * 66}")
    text = s['generated_text']
    # Wrap at 70 chars for readability
    import textwrap
    print(textwrap.fill(text, width=70, initial_indent='  ', subsequent_indent='  '))
    print()

## Stage 04 — Construct seven labeled split Parquets

Stage 04 takes the outputs of Stages 01-03 and assembles seven labeled, leakage-free split Parquets that the downstream training, calibration, and evaluation stages consume.

**Outputs** (written to `paths.splits_dir`, i.e. `pharma_data/splits/`):
* `train.parquet` — MPIB train + 80% benign-clinical
* `val.parquet` — MPIB val + 10% benign-clinical
* `test_v1.parquet` — MPIB test V1-only (headline V1 eval)
* `test_v2.parquet` — MPIB test V2-only (headline V2 eval)
* `calibration.parquet` — 5% benign-clinical, held out for FPR threshold selection
* `clinical_benign_holdout.parquet` — 5% benign-clinical, deployment-FPR estimation (C4b)
* `generic_attack_ood.parquet` — deepset + JailbreakBench, never seen in training

Plus `stage_04_manifest.json` with SHA-256 checksums for every artifact (D52). Stage 04 is fast (~30-60s wall-clock) because it's pure data shuffling.


In [ ]:
!cd /content/pharmguard && python scripts/04_construct_splits.py


### Inspect Stage 04 outputs

Quick check of the seven Parquets. Verifies the D42 fix (`harm_types` deserializes as a Python list, not a JSON string) and prints per-split class balance.


In [ ]:
import pandas as pd
from pathlib import Path
import json

SPLITS = Path('/content/drive/MyDrive/pharmguard/pharma_data/splits')
manifest = json.loads((SPLITS / 'stage_04_manifest.json').read_text())

print(f"Manifest: stage={manifest['stage']}, version={manifest['package_version']}")
print(f"Total rows across all splits: {manifest['row_count_total']}")
print()
print(f"{'Split':<32} {'Rows':>6} {'Benign':>7} {'Adv':>5} {'D42 list?':>10}")
print('-' * 70)
for split_name, rec in manifest['artifacts'].items():
    df = pd.read_parquet(rec['path'])
    n_benign = (df['label'] == 0).sum()
    n_adv = (df['label'] == 1).sum()
    # D42 fix check: harm_types must be a list-like, not a string
    first_ht = df['harm_types'].iloc[0] if len(df) > 0 else None
    is_list = isinstance(first_ht, (list, tuple)) or (hasattr(first_ht, '__iter__') and not isinstance(first_ht, str))
    print(f"{split_name:<32} {len(df):>6} {n_benign:>7} {n_adv:>5} {str(is_list):>10}")


### Stage 04 integration checks

Reads the Stage 04 outputs from Drive (bypassing the manifest) and runs explicit assertions that catch the bug classes we have actually encountered:

* **Schema-level** — every Parquet has the expected columns; `harm_types` deserializes as a list, not a JSON string (the D42 class).
* **Distribution-level** — every split has rows; `train` and `val` have both classes; design-pure splits stay pure (the D53 class).
* **Leakage-level** — no `parent_id` appears in two splits.
* **Upstream contract** — what Stage 05 needs to be true.

Each check prints PASS or FAIL with diagnostic context. The cell never raises; it collects every failure and shows them all at the end. If anything fails, do not proceed to Stage 05 until it is resolved.


In [ ]:
"""Stage 04 post-stage integration checks.

Reads what's actually on disk in pharma_data/splits/, runs a battery
of assertions designed around the bug classes we've hit in development,
and reports PASS/FAIL with diagnostic context.
"""
import pandas as pd
from pathlib import Path
from collections import defaultdict

SPLITS = Path('/content/drive/MyDrive/pharmguard/pharma_data/splits')
EXPECTED_SPLITS = [
    'train', 'val', 'test_v1', 'test_v2',
    'calibration', 'clinical_benign_holdout', 'generic_attack_ood',
]
EXPECTED_COLUMNS = [
    'instance_id', 'parent_id', 'split', 'input_text', 'label',
    'vector', 'source', 'scenario', 'severity', 'harm_types',
    'generation_status', 'wrapper_template_id', 'input_text_hash',
]

failures = []
checks_run = 0

def check(condition, name, detail=''):
    """Record one assertion. Never raises."""
    global checks_run
    checks_run += 1
    if not condition:
        failures.append((name, detail))
        print(f'  FAIL  {name}: {detail}')
    else:
        print(f'  PASS  {name}')

# Load every split once
print('=' * 70)
print('Stage 04 integration checks')
print('=' * 70)

frames = {}
for name in EXPECTED_SPLITS:
    p = SPLITS / f'{name}.parquet'
    if p.exists():
        frames[name] = pd.read_parquet(p)
    else:
        failures.append((f'split_{name}_present', f'{p} missing'))

print(f'\nLoaded {len(frames)} of {len(EXPECTED_SPLITS)} splits')
print()

# ─── Section 1: Schema-level (D42 class) ───
print('--- Schema-level checks ---')
for name, df in frames.items():
    missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    check(
        not missing_cols,
        f'{name}.columns_complete',
        f'missing {missing_cols}' if missing_cols else '',
    )
    if len(df) == 0:
        continue
    # D42 fix: harm_types must be list-like, not a string
    first_ht = df['harm_types'].iloc[0]
    is_listlike = hasattr(first_ht, '__iter__') and not isinstance(first_ht, str)
    check(
        is_listlike,
        f'{name}.harm_types_is_list',
        f'got {type(first_ht).__name__}={first_ht!r}' if not is_listlike else '',
    )
    # Label must be 0 or 1
    bad_labels = set(df['label'].unique()) - {0, 1}
    check(
        not bad_labels,
        f'{name}.labels_binary',
        f'unexpected labels: {bad_labels}' if bad_labels else '',
    )
    # No empty input_text
    empty_texts = (df['input_text'].fillna('').str.len() == 0).sum()
    check(
        empty_texts == 0,
        f'{name}.input_text_nonempty',
        f'{empty_texts} empty input_text rows' if empty_texts else '',
    )
    # No empty parent_id
    empty_pids = (df['parent_id'].fillna('').str.len() == 0).sum()
    check(
        empty_pids == 0,
        f'{name}.parent_id_nonempty',
        f'{empty_pids} empty parent_id rows' if empty_pids else '',
    )

# ─── Section 2: Distribution-level (D53 class) ───
print('\n--- Distribution-level checks ---')

# Train and val must have both classes
MIXED_SPLITS = {'train', 'val'}
for name in MIXED_SPLITS:
    if name not in frames:
        continue
    df = frames[name]
    n_benign = int((df['label'] == 0).sum())
    n_adv = int((df['label'] == 1).sum())
    check(
        n_benign > 0 and n_adv > 0,
        f'{name}.both_classes_present',
        f'benign={n_benign} adversarial={n_adv}',
    )

# Design-pure adversarial splits
PURE_ADV = {'test_v1', 'test_v2', 'generic_attack_ood'}
for name in PURE_ADV:
    if name not in frames:
        continue
    df = frames[name]
    n_benign = int((df['label'] == 0).sum())
    check(
        n_benign == 0,
        f'{name}.adversarial_only',
        f'found {n_benign} benign rows (should be 0)' if n_benign else '',
    )

# Design-pure benign splits
PURE_BENIGN = {'calibration', 'clinical_benign_holdout'}
for name in PURE_BENIGN:
    if name not in frames:
        continue
    df = frames[name]
    n_adv = int((df['label'] == 1).sum())
    check(
        n_adv == 0,
        f'{name}.benign_only',
        f'found {n_adv} adversarial rows (should be 0)' if n_adv else '',
    )

# test_v1 is V1-only, test_v2 is V2-only
if 'test_v1' in frames:
    bad_vec = (frames['test_v1']['vector'] != 'V1').sum()
    check(
        bad_vec == 0,
        'test_v1.vector_v1_only',
        f'{bad_vec} non-V1 rows' if bad_vec else '',
    )
if 'test_v2' in frames:
    bad_vec = (frames['test_v2']['vector'] != 'V2').sum()
    check(
        bad_vec == 0,
        'test_v2.vector_v2_only',
        f'{bad_vec} non-V2 rows' if bad_vec else '',
    )

# ─── Section 3: Leakage (the silent class) ───
print('\n--- Leakage checks ---')
parent_to_splits = defaultdict(set)
for name, df in frames.items():
    for pid in df['parent_id']:
        parent_to_splits[pid].add(name)
leaked = {pid: splits for pid, splits in parent_to_splits.items() if len(splits) > 1}
check(
    not leaked,
    'cross_split.no_parent_id_leakage',
    f'{len(leaked)} parent_ids in multiple splits; first 3: {dict(list(leaked.items())[:3])}' if leaked else '',
)

# ─── Section 4: Upstream contract (Stage 05 needs) ───
print('\n--- Upstream contract (Stage 05) ---')
for name, df in frames.items():
    if len(df) == 0:
        continue
    # input_text under a sanity bound (Stage 05 tokenizer truncates anyway,
    # but >100k chars suggests a different bug)
    p95_len = int(df['input_text'].str.len().quantile(0.95))
    max_len = int(df['input_text'].str.len().max())
    check(
        max_len < 100_000,
        f'{name}.input_text_sane_length',
        f'max={max_len} chars; p95={p95_len}' if max_len >= 100_000 else '',
    )
    # At least 95% of inputs start with [system]
    pct_with_system = (df['input_text'].str.startswith('[system]').sum() / len(df))
    check(
        pct_with_system >= 0.95,
        f'{name}.input_text_has_system_prefix',
        f'only {pct_with_system*100:.1f}% start with [system]' if pct_with_system < 0.95 else '',
    )

# ─── Summary ───
print()
print('=' * 70)
if failures:
    print(f'INTEGRATION CHECKS: {len(failures)} FAILURE(S) out of {checks_run} checks')
    print('=' * 70)
    print('Do not proceed to Stage 05 until these are resolved.')
    for name, detail in failures:
        print(f'  FAIL  {name}: {detail}')
else:
    print(f'INTEGRATION CHECKS: ALL {checks_run} PASSED ✓')
    print('=' * 70)
    print('Stage 04 outputs are safe to consume by Stage 05.')


## Stage 05 — Tokenize with PubMedBERT

Pre-tokenizes every Stage 04 split so Stage 06's training loop does not re-tokenize per batch. Right-truncation at 512 tokens (D56), with a measurement gate that surfaces per-vector V2 truncation rates so we can revisit the truncation strategy if V2 attacks systematically get cut off.

Runs in 1-3 minutes wall-clock on Colab (PubMedBERT's fast tokenizer is CPU-bound but parallelized internally).


In [ ]:
!cd /content/pharmguard && python scripts/05_tokenize.py


### Inspect Stage 05 outputs

Quick look at one tokenized split. Verifies the D42 fix (input_ids deserializes as a Python list, not a JSON string), shows the truncation pattern, and prints per-vector V2 truncation rates against the D56 measurement gate.


In [ ]:
import pandas as pd
import json
from pathlib import Path

TOKENIZED = Path('/content/drive/MyDrive/pharmguard/pharma_data/tokenized')
manifest = json.loads((TOKENIZED / 'stage_05_manifest.json').read_text())

print(f"Tokenizer: {manifest['tokenizer_name']}")
print(f"Max length: {manifest['max_length']}")
print(f"Truncation: {manifest['truncation_direction']}")
print()
print(f"{'Split':<32} {'Rows':>6} {'Trunc%':>7} {'Mean':>5} {'P95':>5} {'Max':>5}")
print('-' * 70)
for split_name, stats in manifest['stats_by_split'].items():
    if stats['rows'] == 0:
        continue
    print(f"{split_name:<32} {stats['rows']:>6} "
          f"{100*stats['truncation_rate']:>6.1f}% "
          f"{stats['mean_token_count']:>5.0f} "
          f"{stats['p95_token_count']:>5} "
          f"{stats['max_token_count']:>5}")

print()
print('Per-vector V2 truncation rates (D56 measurement gate):')
print(f"  threshold = 25.0% — warn if exceeded")
for split_name, stats in manifest['stats_by_split'].items():
    v2 = stats.get('by_vector', {}).get('V2')
    if v2 is None or v2['rows'] == 0:
        continue
    rate = v2['truncation_rate']
    marker = '⚠ EXCEEDS' if rate > 0.25 else '  OK'
    print(f"  {marker:<10} {split_name:<32} V2 truncation: "
          f"{100*rate:.1f}% ({v2['truncated_rows']}/{v2['rows']})")

# Show one tokenized row
print()
print('Sample tokenized row from train.parquet:')
df = pd.read_parquet(TOKENIZED / 'train.parquet')
row = df.iloc[0]
print(f"  instance_id:    {row['instance_id']}")
print(f"  label:          {row['label']}")
print(f"  vector:         {row['vector']}")
print(f"  token_count:    {row['token_count']}")
print(f"  was_truncated:  {row['was_truncated']}")
print(f"  input_ids[:10]: {list(row['input_ids'][:10])}")
print(f"  D42 check (input_ids is list-like, not str): {not isinstance(row['input_ids'], str)}")


### Stage 05 integration checks

Same pattern as the Stage 04 integration cell (D54). Reads tokenized outputs directly from Drive and runs explicit assertions. Catches schema drift, D42-class regressions, and structural inconsistencies (e.g., `len(input_ids) != len(attention_mask)`). Never raises; collects every failure and shows them all.


In [ ]:
"""Stage 05 post-stage integration checks.

Runs assertions designed around the bug classes we've hit in development.
Reads from Drive directly; bypasses the manifest. Never raises.
"""
import pandas as pd
import json
from pathlib import Path

TOKENIZED = Path('/content/drive/MyDrive/pharmguard/pharma_data/tokenized')
EXPECTED_SPLITS = [
    'train', 'val', 'test_v1', 'test_v2',
    'calibration', 'clinical_benign_holdout', 'generic_attack_ood',
]
EXPECTED_COLUMNS = [
    'instance_id', 'parent_id', 'split', 'input_text', 'input_text_hash',
    'input_ids', 'attention_mask', 'token_count', 'was_truncated',
    'truncated_token_count', 'label', 'vector', 'source', 'scenario',
    'severity', 'harm_types', 'generation_status', 'wrapper_template_id',
]
MAX_LENGTH = 512

failures = []
checks_run = 0

def check(condition, name, detail=''):
    global checks_run
    checks_run += 1
    if not condition:
        failures.append((name, detail))
        print(f'  FAIL  {name}: {detail}')
    else:
        print(f'  PASS  {name}')

print('=' * 70)
print('Stage 05 integration checks')
print('=' * 70)

frames = {}
for name in EXPECTED_SPLITS:
    p = TOKENIZED / f'{name}.parquet'
    if p.exists():
        frames[name] = pd.read_parquet(p)
    else:
        failures.append((f'split_{name}_present', f'{p} missing'))

print(f'\nLoaded {len(frames)} of {len(EXPECTED_SPLITS)} splits')
print()

# --- Section 1: Schema (D42 class) ---
print('--- Schema-level checks ---')
for name, df in frames.items():
    missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    check(not missing_cols, f'{name}.columns_complete',
          f'missing {missing_cols}' if missing_cols else '')
    if len(df) == 0:
        continue
    # D42: input_ids must be list-like
    first_ids = df['input_ids'].iloc[0]
    is_list = hasattr(first_ids, '__iter__') and not isinstance(first_ids, str)
    check(is_list, f'{name}.input_ids_is_list',
          f'got {type(first_ids).__name__}' if not is_list else '')
    # attention_mask same check
    first_mask = df['attention_mask'].iloc[0]
    is_list = hasattr(first_mask, '__iter__') and not isinstance(first_mask, str)
    check(is_list, f'{name}.attention_mask_is_list',
          f'got {type(first_mask).__name__}' if not is_list else '')

# --- Section 2: Structural consistency ---
print('\n--- Structural consistency ---')
for name, df in frames.items():
    if len(df) == 0:
        continue
    # len(input_ids) == len(attention_mask)
    mismatches = (df['input_ids'].apply(len) != df['attention_mask'].apply(len)).sum()
    check(mismatches == 0, f'{name}.input_ids_mask_lengths_match',
          f'{mismatches} rows mismatch' if mismatches else '')
    # token_count == len(input_ids)
    tc_mismatches = (df['input_ids'].apply(len) != df['token_count']).sum()
    check(tc_mismatches == 0, f'{name}.token_count_matches_input_ids',
          f'{tc_mismatches} rows mismatch' if tc_mismatches else '')
    # No row exceeds MAX_LENGTH
    max_obs = int(df['token_count'].max())
    check(max_obs <= MAX_LENGTH, f'{name}.token_count_within_max',
          f'max={max_obs} > {MAX_LENGTH}' if max_obs > MAX_LENGTH else '')
    # was_truncated implies truncated_token_count > 0
    trunc_with_zero = ((df['was_truncated']) & (df['truncated_token_count'] == 0)).sum()
    check(trunc_with_zero == 0, f'{name}.truncated_count_consistent',
          f'{trunc_with_zero} truncated rows with 0 truncated_token_count' if trunc_with_zero else '')

# --- Section 3: D56 measurement gate ---
print('\n--- D56 measurement gate (per-vector V2 truncation) ---')
for name, df in frames.items():
    if len(df) == 0:
        continue
    v2 = df[df['vector'] == 'V2']
    if len(v2) == 0:
        continue
    rate = float(v2['was_truncated'].mean())
    # NOT a failure — this is informational. We print PASS regardless.
    # The check exists to ensure the gate is being measured per split.
    check(True, f'{name}.v2_truncation_measured',
          f'rate={100*rate:.1f}%')

# --- Section 4: Class balance preserved from Stage 04 ---
print('\n--- Class balance preserved ---')
MIXED_SPLITS = {'train', 'val'}
for name in MIXED_SPLITS:
    if name not in frames:
        continue
    df = frames[name]
    n_benign = int((df['label'] == 0).sum())
    n_adv = int((df['label'] == 1).sum())
    check(n_benign > 0 and n_adv > 0, f'{name}.both_classes_present',
          f'benign={n_benign} adversarial={n_adv}')

# --- Summary ---
print()
print('=' * 70)
if failures:
    print(f'INTEGRATION CHECKS: {len(failures)} FAILURE(S) out of {checks_run} checks')
    print('=' * 70)
    print('Do not proceed to Stage 06 until these are resolved.')
    for name, detail in failures:
        print(f'  FAIL  {name}: {detail}')
else:
    print(f'INTEGRATION CHECKS: ALL {checks_run} PASSED ✓')
    print('=' * 70)
    print('Stage 05 outputs are safe to consume by Stage 06.')


## Stage 06A — Single-seed training (scaffold)

First training run. Trains the PharmGuard model (PubMedBERT + classification head) with class-weighted cross-entropy on `train.parquet`, validates on `val.parquet` every epoch, keeps the checkpoint with best val AUC (D66), then calibrates the decision threshold against the pure-benign `calibration.parquet` at three target FPRs (1%, 0.5%, 0.1%) per D67.

Output: `pharma_models/seed_42_config_main/` containing the trained weights, tokenizer, training log, calibration JSON, and a SHA-256 manifest.

**Wall-clock: 30-45 min on L4.** Validates the training loop end-to-end before we scale to multi-seed (Stage 06B). If anything fails — wrong loss, wrong metric, OOM, determinism broken — we catch it here at the cheapest possible point.


In [ ]:
!cd /content/pharmguard && python scripts/06A_train_single_seed.py --seed 42 --config main


### Inspect Stage 06A outputs

Read the manifest and per-epoch training log. The headline number is **best_val_auc** — anything above ~0.85 means the model is learning meaningful discrimination. Below ~0.75 suggests a real problem (label noise, loss not converging, learning rate way off).


In [ ]:
import json
from pathlib import Path

MODEL_DIR = Path('/content/drive/MyDrive/pharmguard/pharma_models/seed_42_config_main')
manifest = json.loads((MODEL_DIR / 'stage_06A_manifest.json').read_text())

print(f"Package version:    {manifest['package_version']}")
print(f"Seed:               {manifest['seed']}")
print(f"Config:             {manifest['config_name']}")
print(f"Encoder:            {manifest['encoder_name']}")
print(f"Best epoch:         {manifest['best_epoch']}")
print(f"Best val AUC:       {manifest['best_val_auc']:.4f}")
print()
print('Per-epoch training log:')
print(f"{'epoch':>6} {'val_auc':>8} {'val_loss':>9} {'val_f1':>7} {'val_precision':>14} {'val_recall':>11}")
for entry in manifest['training_log']:
    print(f"  {entry['epoch']:>4} {entry['eval_auc']:>8.4f} "
          f"{entry['eval_loss']:>9.4f} {entry['eval_f1']:>7.4f} "
          f"{entry['eval_precision']:>14.4f} {entry['eval_recall']:>11.4f}")
print()
print('Calibration (FPR → threshold + Wilson 95% CI):')
for fpr_key, rec in manifest['calibration']['thresholds'].items():
    print(f"  {fpr_key:<14} τ = {rec['threshold']:.4f}  "
          f"realized = {rec['realized_fpr']:.4f}  "
          f"CI = [{rec['wilson_ci_low']:.4f}, {rec['wilson_ci_high']:.4f}]")
print()
print('Benign-score distribution on calibration set:')
dist = manifest['calibration']['score_distribution']
print(f"  min={dist['min']:.4f}  p50={dist['p50']:.4f}  p90={dist['p90']:.4f}  "
      f"p99={dist['p99']:.4f}  max={dist['max']:.4f}")
print(f"  mean={dist['mean']:.4f}  std={dist['std']:.4f}")
print()
print(f"Artifacts (SHA-256 verified at write time):")
for name, rec in manifest['artifacts'].items():
    print(f"  {name:<32} {rec['size_bytes']:>10,} bytes  sha256={rec['sha256'][:12]}...")


### Stage 06A integration checks

Same D54 pattern. Read the manifest and artifacts directly, run explicit assertions on schema, training-log shape, calibration validity, and sanity bounds on the headline metric. Never raises; collects all failures and reports at the end.


In [ ]:
"""Stage 06A integration checks.

Verifies the training run produced a usable model with sensible
metrics and a valid calibration. Catches: missing artifacts,
manifest checksum drift, training-log shape problems, calibration
thresholds outside [0, 1], val AUC below random chance.
"""
import json
import hashlib
from pathlib import Path

MODEL_DIR = Path('/content/drive/MyDrive/pharmguard/pharma_models/seed_42_config_main')
manifest_path = MODEL_DIR / 'stage_06A_manifest.json'

failures = []
checks_run = 0

def check(condition, name, detail=''):
    global checks_run
    checks_run += 1
    if not condition:
        failures.append((name, detail))
        print(f'  FAIL  {name}: {detail}')
    else:
        print(f'  PASS  {name}')

print('=' * 70)
print('Stage 06A integration checks')
print('=' * 70)

check(manifest_path.exists(), 'manifest_exists',
      f'missing: {manifest_path}')
if not manifest_path.exists():
    print('Cannot proceed without manifest.')
else:
    manifest = json.loads(manifest_path.read_text())
    
    print()
    print('--- Manifest schema ---')
    required = {'stage', 'package_version', 'seed', 'config_name',
                'encoder_name', 'best_epoch', 'best_val_auc',
                'calibration', 'training_log', 'artifacts'}
    missing = required - set(manifest)
    check(not missing, 'manifest_has_required_keys',
          f'missing keys: {missing}' if missing else '')
    check(manifest.get('stage') == 'stage_06A_train_single_seed',
          'manifest_stage_correct')
    
    print()
    print('--- Artifact checksums ---')
    def sha256_of(path):
        h = hashlib.sha256()
        with open(path, 'rb') as f:
            for chunk in iter(lambda: f.read(1<<20), b''):
                h.update(chunk)
        return h.hexdigest()
    
    for name, rec in manifest['artifacts'].items():
        p = Path(rec['path'])
        check(p.exists(), f'artifact_{name}_present', f'{p}')
        if p.exists():
            actual = sha256_of(p)
            check(actual == rec['sha256'],
                  f'artifact_{name}_checksum_match',
                  f'expected {rec["sha256"][:12]}, got {actual[:12]}')
    
    # Required model artifacts — these are the files Stage 06B/06C/07 will read
    required_artifacts = {'config.json', 'training_log.json',
                          'calibration.json'}
    has_required = required_artifacts & set(manifest['artifacts'])
    check(has_required == required_artifacts,
          'required_artifacts_present',
          f'missing: {required_artifacts - has_required}')
    
    # Either pytorch_model.bin or model.safetensors
    has_weights = any(
        n in manifest['artifacts']
        for n in ('pytorch_model.bin', 'model.safetensors')
    )
    check(has_weights, 'model_weights_present',
          'expected pytorch_model.bin or model.safetensors')
    
    print()
    print('--- Training log ---')
    tlog = manifest.get('training_log', [])
    check(len(tlog) > 0, 'training_log_nonempty', f'log has {len(tlog)} entries')
    for i, entry in enumerate(tlog):
        check('eval_auc' in entry, f'training_log[{i}]_has_auc')
        if 'eval_auc' in entry:
            auc = entry['eval_auc']
            check(0.0 <= auc <= 1.0, f'training_log[{i}]_auc_valid',
                  f'val_auc={auc} out of [0,1]')
    
    print()
    print('--- Calibration ---')
    cal = manifest['calibration']
    check('thresholds' in cal, 'calibration_has_thresholds')
    for fpr_key, rec in cal.get('thresholds', {}).items():
        tau = rec['threshold']
        check(0.0 <= tau <= 1.0 + 1e-6, f'{fpr_key}_threshold_valid',
              f'threshold={tau} out of [0,1]')
        rfpr = rec['realized_fpr']
        check(0.0 <= rfpr <= 1.0, f'{fpr_key}_realized_fpr_valid',
              f'realized_fpr={rfpr} out of [0,1]')
        ci_low, ci_high = rec['wilson_ci_low'], rec['wilson_ci_high']
        check(0.0 <= ci_low <= ci_high <= 1.0,
              f'{fpr_key}_wilson_ci_valid',
              f'CI=[{ci_low}, {ci_high}]')
    
    print()
    print('--- Sanity bounds on headline metric ---')
    best_auc = manifest['best_val_auc']
    # Random-chance AUC is 0.5; any model trained on 14k labeled rows
    # for 3 epochs should beat this comfortably.
    check(best_auc > 0.55, 'best_val_auc_above_random',
          f'best_val_auc={best_auc:.4f} — model may have failed to learn')
    if best_auc > 0.55:
        if best_auc > 0.95:
            print(f'  NOTE: best_val_auc={best_auc:.4f} is very high — '
                  f'verify class-balance is realistic, not 50/50')
        elif best_auc < 0.80:
            print(f'  NOTE: best_val_auc={best_auc:.4f} is below '
                  f'PromptShield baselines on similar data — investigate')

print()
print('=' * 70)
if failures:
    print(f'INTEGRATION CHECKS: {len(failures)} FAILURE(S) of {checks_run}')
    print('=' * 70)
    print('Do not proceed to Stage 06B until these are resolved.')
    for name, detail in failures:
        print(f'  FAIL  {name}: {detail}')
else:
    print(f'INTEGRATION CHECKS: ALL {checks_run} PASSED ✓')
    print('=' * 70)
    print('Single-seed scaffold validated. Safe to scale to multi-seed (Stage 06B).')


In [ ]:
"""
Drive-state inspection cell (added after v0.8.0 data-loss event).

Runs after the full Stage 06A cycle to confirm that all expected
artifacts have persisted to Drive. If anything is missing here, it
is missing on Drive and will not survive a session restart.

Use this cell:
  * Immediately after Stage 06A completes (before assuming the model
    is safe).
  * On a fresh session, to check what was preserved.
  * Whenever Drive sync is suspect.
"""
import sys
from pathlib import Path

sys.path.insert(0, '/content/pharmguard')

from pharmguard.data.drive_sync import (
    force_drive_sync,
    verify_directory_persisted,
    DriveSyncError,
)

DRIVE_ROOT = Path('/content/drive/MyDrive/pharmguard')

print(f"{'=' * 60}")
print(f"Drive-state inspection")
print(f"{'=' * 60}")
print(f"  Drive root: {DRIVE_ROOT}")
print(f"  (forcing sync + 5s wait before listing...)")
force_drive_sync(wait_s=5.0)

# Expected directory structure after a complete run
expected = {
    'pharma_data/raw': None,         # MPIB raw (~80 MB)
    'pharma_data/processed': ['parsed_mpib.parquet'],
    'pharma_data/benign':    ['all_benign.csv'],
    'pharma_data/adversarial': ['reconstructed_v2.parquet'],
    'pharma_data/splits':    [
        'train.parquet', 'val.parquet',
        'test_v1.parquet', 'test_v2.parquet',
        'calibration.parquet', 'clinical_benign_holdout.parquet',
        'generic_attack_ood.parquet',
        'stage_04_manifest.json',
    ],
    'pharma_data/tokenized': [
        'train.parquet', 'val.parquet',
        'test_v1.parquet', 'test_v2.parquet',
        'calibration.parquet', 'clinical_benign_holdout.parquet',
        'generic_attack_ood.parquet',
        'stage_05_manifest.json',
    ],
    'pharma_models/seed_42_config_main': [
        'model.safetensors', 'config.json',
        'tokenizer.json', 'tokenizer_config.json',
        'calibration.json', 'training_log.json',
        'stage_06A_manifest.json',
    ],
}

all_ok = True
for rel, files in expected.items():
    dpath = DRIVE_ROOT / rel
    if not dpath.exists():
        print(f"  ✗ MISSING: {rel}/")
        all_ok = False
        continue
    if files is None:
        # Just check directory exists; don't enumerate contents
        n_files = sum(1 for _ in dpath.iterdir() if _.is_file())
        print(f"  ✓ {rel}/  ({n_files} files)")
        continue
    try:
        verify_directory_persisted(dpath, files, wait_s=1.0)
        total_mb = sum(
            (dpath / f).stat().st_size for f in files
        ) / 1024 / 1024
        print(f"  ✓ {rel}/  ({len(files)} files, {total_mb:.1f} MB)")
    except DriveSyncError as e:
        print(f"  ✗ {rel}/  → {e}")
        all_ok = False

print(f"{'=' * 60}")
if all_ok:
    print(f"  All expected artifacts present on Drive ✓")
else:
    print(f"  SOME ARTIFACTS MISSING — see entries above")
    print(f"  Affected stages need rerun before relying on these outputs")
print(f"{'=' * 60}")


---

After Stage 06A passes integration checks: Stage 06B runs the same training across all 5 seeds in CONFIG.seeds. Stage 06C runs the four ablations (BERT-base, DistilBERT, no class weights, no benign-clinical wrappers) on a single seed each. Stage 07 then evaluates every saved model against the four published baselines.
